# Coupled oscillators and normal modes

*mechanics · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/mechanics/coupled_oscillators)

$N$ equal masses joined by springs between two fixed walls. The springs become one stiffness tensor $K_{ij}$, its eigenvectors are the normal modes, and the motion is a superposition of modes summed with a single einsum.

**Vector form**

$$m\,\ddot{\mathbf x} = -\mathsf K\,\mathbf x$$

**Index (tensor) form**

$$m\,\ddot x_i = -K_{ij}\,x_j,\qquad x_i(t) = \sum_n A_n\,e_{in}\cos(\omega_n t + \varphi_n)$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_i$ | `x[t, i]` | (T, N) | displacement of mass $i$ from its rest position |
| $K_{ij}$ | `stiffness(n, k)` | (N, N) | stiffness tensor: $2k$ on the diagonal, $-k$ next to it |
| $K_{ij}x_j$ | `einsum('ij,...j->...i', K, x)` | (N,) | repeated $j$ ⇒ summed: net spring force on mass $i$ |
| $\omega_n^2,\ e_{in}$ | `torch.linalg.eigh(K / m)` | (N,), (N, N) | eigenvalues and eigenvectors (column $n$) = normal modes |
| $q_n = e_{in}x_i$ | `einsum('in,i->n', e, x0)` | (N,) | repeated $i$ ⇒ summed: project the start onto each mode |
| $x_i(t)$ | `einsum('in,tn->ti', e, modal)` | (T, N) | repeated $n$ ⇒ summed: superposition of modes |

Because $K_{ij}$ is symmetric, eigh gives real frequencies and orthonormal modes, so every motion is a sum of independent cosines and no time stepping is needed. Start in a single mode and the chain keeps that shape forever; pluck one mass and the energy sloshes along the chain. The analytic frequencies are $\omega_n = 2\sqrt{k/m}\,\sin\tfrac{n\pi}{2(N+1)}$.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/mechanics/coupled_oscillators/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    n, k, m = params["n_masses"], params["k"], params["m"]
    K = tensors.stiffness(n, k, dev)
    x0 = tensors.initial_displacement(n, params["start"], params["amplitude"], dev)

    sol = solve.solve(K, m, x0, params["t_max"])
    kinetic, potential = equation.energy(sol["x"], sol["v"], K, m)

    exact = tensors.analytic_frequencies(n, k, m).tolist()
    for p, (w, a) in enumerate(zip(sol["omega"].tolist(), sol["A"].tolist())):
        print(f"mode {p + 1:2d}  omega = {w:6.3f} rad/s  (analytic {exact[p]:6.3f})  amplitude {a:6.3f} m")
    return plot.animate(sol, kinetic, potential, out_dir / "coupled_oscillators.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/mechanics/coupled_oscillators/equation.py
r"""Physics → tensor translation.

Newton's second law for N masses m joined by springs k between two fixed walls,

    m \ddot{\mathbf x} = -\mathsf K \mathbf x,

reads in index notation (i, j = 1 … N label the masses)

    m \ddot x_i = -K_{ij} x_j

The repeated index j is summed: each mass feels the springs of its neighbours, einsum('ij,j->i').

A normal mode is a motion where every mass oscillates with one frequency, x_i = e_{in} \cos(\omega_n t).
Plugging that in gives the eigenvalue problem (no sum over n here)

    (K_{ij} / m) \, e_{jn} = \omega_n^2 \, e_{in}

and the general motion is a superposition, summed over the mode index n:

    x_i(t) = \sum_n A_n e_{in} \cos(\omega_n t + \varphi_n)
"""

import torch


def spring_force(K: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    r"""F_i = -K_{ij} x_j.   K: (N, N), x: (..., N) → (..., N)"""
    return -torch.einsum("ij,...j->...i", K, x)


def normal_modes(K: torch.Tensor, m: float) -> tuple[torch.Tensor, torch.Tensor]:
    r"""Solve (K_{ij}/m) e_{jn} = \omega_n^2 e_{in}.   Returns ω (N,) and e (N, N); mode n is the column e[:, n]."""
    omega_sq, e = torch.linalg.eigh(K / m)  # K symmetric ⇒ real ω², orthonormal eigenvectors
    return torch.sqrt(omega_sq), e


def mode_amplitudes(
    e: torch.Tensor, omega: torch.Tensor, x0: torch.Tensor, v0: torch.Tensor
) -> tuple[torch.Tensor, torch.Tensor]:
    r"""A_n and \varphi_n from the start state, via the modal coordinates q_n = e_{in} x_i."""
    q0 = torch.einsum("in,i->n", e, x0)  # (N,)  q_n(0), sum over masses i
    q0_dot = torch.einsum("in,i->n", e, v0)  # (N,)  \dot q_n(0)
    A = torch.sqrt(q0**2 + (q0_dot / omega) ** 2)  # q_n(t) = A_n cos(ω_n t + φ_n)
    phi = torch.atan2(-q0_dot / omega, q0)
    return A, phi


def superpose(
    e: torch.Tensor, omega: torch.Tensor, A: torch.Tensor, phi: torch.Tensor, t: torch.Tensor
) -> tuple[torch.Tensor, torch.Tensor]:
    r"""x_{ti} = e_{in} A_n \cos(\omega_n t + \varphi_n) and its time derivative v_{ti}.   t: (T,) → (T, N) each."""
    angle = torch.einsum("t,n->tn", t, omega) + phi  # (T, N)  ω_n t + φ_n
    x = torch.einsum("in,tn->ti", e, A * torch.cos(angle))  # repeated n ⇒ sum over modes
    v = torch.einsum("in,tn->ti", e, -A * omega * torch.sin(angle))
    return x, v


def energy(x: torch.Tensor, v: torch.Tensor, K: torch.Tensor, m: float) -> tuple[torch.Tensor, torch.Tensor]:
    r"""Kinetic ½ m v_i v_i and potential ½ x_i K_{ij} x_j (all indices summed).   (..., N) → (...,) each."""
    kinetic = 0.5 * m * torch.einsum("...i,...i->...", v, v)
    potential = 0.5 * torch.einsum("...i,ij,...j->...", x, K, x)
    return kinetic, potential


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/mechanics/coupled_oscillators/tensors.py
"""Build the concrete tensors: the stiffness tensor K_ij and the initial displacement x_i(0)."""

import math

import torch


def stiffness(n: int, k: float, device: torch.device | None = None) -> torch.Tensor:
    """K_ij of shape (N, N) for the chain  wall – m – m – … – m – wall.

    Mass i is pulled by the springs on both sides (2k on the diagonal) and
    coupled to its neighbours i ± 1 (−k on the off-diagonals):

        K = k [[ 2, -1,  0, …],
               [-1,  2, -1, …],
               [ 0, -1,  2, …], …]
    """
    diagonal = 2 * k * torch.ones(n, device=device)  # (N,)
    off = -k * torch.ones(n - 1, device=device)  # (N-1,)
    return torch.diag(diagonal) + torch.diag(off, 1) + torch.diag(off, -1)


def analytic_frequencies(n: int, k: float, m: float) -> torch.Tensor:
    """Textbook result ω_n = 2 √(k/m) sin(nπ / 2(N+1)) for n = 1 … N: shape (N,)."""
    modes = torch.arange(1, n + 1)
    return 2 * math.sqrt(k / m) * torch.sin(modes * math.pi / (2 * (n + 1)))


def initial_displacement(n: int, start: str, amplitude: float, device: torch.device | None = None) -> torch.Tensor:
    """x_i(0) of shape (N,). The masses always start at rest, v_i(0) = 0."""
    if start == "first mass":
        x0 = torch.zeros(n, device=device)
        x0[0] = amplitude
    elif start == "middle mass":
        x0 = torch.zeros(n, device=device)
        x0[n // 2] = amplitude
    else:  # one standing wave  sin(p π i / (N+1)),  i = 1 … N
        p = {"mode 1": 1, "mode 2": 2, "highest mode": n}[start]
        i = torch.arange(1, n + 1, device=device)
        x0 = amplitude * torch.sin(p * math.pi * i / (n + 1))
    return x0


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/mechanics/coupled_oscillators/solve.py
"""Solve the chain exactly: diagonalise K, then add up the normal-mode cosines."""

import torch

from .equation import mode_amplitudes, normal_modes, superpose


def solve(K: torch.Tensor, m: float, x0: torch.Tensor, t_max: float, n_times: int = 1001) -> dict[str, torch.Tensor]:
    """Return times (T,), positions x and velocities v (T, N), and the modes ω (N,), e (N, N), A (N,).

    No time stepping is needed: once the modes are known, x_i(t) is a closed formula
    that we evaluate at all T times in one einsum.
    """
    omega, e = normal_modes(K, m)
    A, phi = mode_amplitudes(e, omega, x0, torch.zeros_like(x0))  # start at rest
    times = torch.linspace(0.0, t_max, n_times, device=K.device)
    x, v = superpose(e, omega, A, phi, times)
    return {"times": times, "x": x, "v": v, "omega": omega, "e": e, "A": A}


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/mechanics/coupled_oscillators/plot.py
"""Animate the chain of masses, with the strongest normal modes and the energy below it."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif


def zigzag(x_start: float, x_end: float, n_coils: int = 4, width: float = 0.12) -> tuple[np.ndarray, np.ndarray]:
    """Points of a spring drawn as a zigzag between x_start and x_end at height 0."""
    xs = np.linspace(x_start, x_end, 2 * n_coils + 3)
    ys = np.zeros_like(xs)
    ys[1:-1:2], ys[2:-1:2] = width, -width
    ys[-2] = 0.0
    return xs, ys


def animate(sol: dict, kinetic: torch.Tensor, potential: torch.Tensor, path: Path) -> Path:
    t = sol["times"].cpu().numpy()  # (T,)
    x = sol["x"].cpu().numpy()  # (T, N)
    e = sol["e"].cpu().numpy()  # (N, N) mode n = column n
    omega, A = sol["omega"].cpu().numpy(), sol["A"].cpu().numpy()  # (N,)
    ke, pe = kinetic.cpu().numpy(), potential.cpu().numpy()  # (T,)
    n = x.shape[1]
    rest = np.arange(1, n + 1)  # equilibrium positions, spacing 1 m; walls at 0 and N+1

    fig = plt.figure(figsize=(7, 4.6))
    grid = fig.add_gridspec(2, 2, height_ratios=[1, 1.3])
    ax_chain = fig.add_subplot(grid[0, :])
    ax_modes = fig.add_subplot(grid[1, 0])
    ax_energy = fig.add_subplot(grid[1, 1])

    # --- top: the chain itself ---
    ax_chain.set_xlim(-0.3, n + 1.3)
    ax_chain.set_ylim(-0.5, 0.5)
    ax_chain.set_yticks([])
    ax_chain.grid(False)
    ax_chain.set_xlabel("position (m)")
    ax_chain.axvspan(-0.25, 0, color="#9ca3af")  # left wall
    ax_chain.axvspan(n + 1, n + 1.25, color="#9ca3af")  # right wall
    springs = [ax_chain.plot([], [], color="#6b7280", lw=1)[0] for _ in range(n + 1)]
    masses = ax_chain.plot([], [], "s", ms=14, color=ACCENT[0])[0]
    clock = ax_chain.text(0.01, 0.92, "", transform=ax_chain.transAxes, va="top", fontsize=9)

    # --- bottom left: shapes of the three most excited modes e_{in} (static) ---
    strongest = np.argsort(A)[::-1][: min(3, n)]
    sites = np.arange(n + 2)
    for row, p in enumerate(strongest):
        shape = np.concatenate([[0.0], e[:, p] * np.sign(e[0, p] or 1.0), [0.0]])  # fixed ends
        offset = -1.2 * row
        ax_modes.axhline(offset, color="#d1d5db", lw=0.8)
        ax_modes.plot(sites, offset + shape, "o-", ms=3, color=ACCENT[(row + 1) % len(ACCENT)])
        ax_modes.text(n + 1.2, offset, f"n={p + 1}\nω={omega[p]:.2f}", fontsize=7, va="center")
    ax_modes.set_xlim(-0.3, n + 3.2)
    ax_modes.set_yticks([])
    ax_modes.set_xlabel("mass index i")
    ax_modes.set_title("strongest modes $e_{in}$", fontsize=9)

    # --- bottom right: energy vs time ---
    ax_energy.plot(t, ke, lw=1, label="kinetic")
    ax_energy.plot(t, pe, lw=1, label="potential")
    ax_energy.plot(t, ke + pe, lw=1.8, color="#111827", label="total")
    cursor = ax_energy.axvline(0, color="#9ca3af", lw=1)
    ax_energy.set_xlim(t[0], t[-1])
    ax_energy.set_ylim(0, (ke + pe).max() * 1.15)
    ax_energy.set_xlabel("t (s)")
    ax_energy.set_ylabel("energy (J)")
    ax_energy.legend(fontsize=7, loc="lower right")

    def update(step: int):
        pos = rest + x[step]  # (N,) where each mass is now
        ends = np.concatenate([[0.0], pos, [n + 1.0]])
        for s, spring in enumerate(springs):
            spring.set_data(*zigzag(ends[s], ends[s + 1]))
        masses.set_data(pos, np.zeros(n))
        cursor.set_xdata([t[step]])
        clock.set_text(f"t = {t[step]:.2f} s")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), n_frames=100), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_masses': 6,
    'k': 10.0,
    'm': 1.0,
    'start': 'first mass',
    'amplitude': 0.3,
    't_max': 10.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.mechanics.coupled_oscillators")]:
    del sys.modules[name]
main = importlib.import_module("problems.mechanics.coupled_oscillators.main")
problem_dir = Path("problems/mechanics/coupled_oscillators")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
